# TinyLlama 1.1B — Portfolio Agent

Fine-tune → Merge → MLC (WebGPU) pipeline.
Runs on Colab free T4. Output: browser-ready WebLLM model.

## 1. Install, clone & build dataset

In [ ]:
import torch, subprocess, os, shutil
from pathlib import Path

os.environ["GIT_TERMINAL_PROMPT"] = "0"

# GPU check
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU"
props = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
mem_gb = props.total_memory / 1e9 if props else 0
print(f"GPU: {device_name}  |  VRAM: {mem_gb:.1f}GB")

# Clone repo
if not Path("pesnik.dev").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pesnik/pesnik.dev.git"], check=True)
os.chdir("pesnik.dev")

# Build dataset
subprocess.run(["python", "scripts/prepare_dataset.py", "--output", "data/train.jsonl"], check=True)

# Install deps
subprocess.run(["pip", "install", "-qU", "transformers", "datasets", "accelerate", "peft", "trl", "bitsandbytes", "huggingface_hub", "torchao"], check=True)
print("\nSetup complete.")

## 2. Fine-tune (LoRA)
~20 min on T4. Auto-detects TRL version.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
LORA_DIR = "pesnik-tinyllama-lora"

dataset = load_dataset("json", data_files="data/train.jsonl", split="train")
print(f"Dataset: {len(dataset)} samples")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
    device_map="auto", torch_dtype=torch.bfloat16)
model.config.use_cache = False
model.gradient_checkpointing_enable()

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

import trl
if hasattr(trl, "SFTConfig"):
    from trl import SFTConfig, SFTTrainer
    args = SFTConfig(output_dir=LORA_DIR, max_length=1024, dataset_text_field="messages",
        packing=False, per_device_train_batch_size=4, gradient_accumulation_steps=2,
        gradient_checkpointing=True, num_train_epochs=3, learning_rate=2e-4, warmup_steps=20,
        logging_steps=10, save_steps=100, save_total_limit=2, bf16=True, report_to="none",
        remove_unused_columns=False)
    trainer = SFTTrainer(model=model, args=args, train_dataset=dataset, processing_class=tokenizer)
else:
    from trl import SFTTrainer
    args = TrainingArguments(output_dir=LORA_DIR, per_device_train_batch_size=4,
        gradient_accumulation_steps=2, gradient_checkpointing=True, num_train_epochs=3,
        learning_rate=2e-4, warmup_steps=20, logging_steps=10, save_steps=100, save_total_limit=2,
        bf16=True, report_to="none", remove_unused_columns=False)
    trainer = SFTTrainer(model=model, args=args, train_dataset=dataset,
        max_seq_length=1024, dataset_text_field="messages", packing=False)
    trainer.tokenizer = tokenizer

trainer.train()
trainer.save_model(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA saved to {LORA_DIR}")

## 3. Push LoRA adapters to HF

In [ ]:
from huggingface_hub import HfApi, notebook_login

LORA_HF_REPO = "pesnik/pesnik-lora"

notebook_login()
api = HfApi()
api.create_repo(LORA_HF_REPO, exist_ok=True)
api.upload_folder(folder_path=LORA_DIR, repo_id=LORA_HF_REPO, repo_type="model")
print(f"LoRA adapters at https://huggingface.co/{LORA_HF_REPO}")

## 4. Merge LoRA → full model & push to HF

In [ ]:
from peft import PeftModel
import torch
from huggingface_hub import HfApi, notebook_login

HF_REPO = "pesnik/pesnik"
MERGED_DIR = "pesnik-merged"

print("Loading base model...")
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

print("Loading LoRA adapters...")
model = PeftModel.from_pretrained(base, LORA_DIR)

print("Merging LoRA into base...")
merged = model.merge_and_unload()

print(f"Saving merged model to {MERGED_DIR}...")
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.save_pretrained(MERGED_DIR)

print("Uploading merged model to HF...")
notebook_login()
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path=MERGED_DIR, repo_id=HF_REPO, repo_type="model")
print(f"Merged model at https://huggingface.co/{HF_REPO}")

## 5. Done

The website uses mlc-ai's pre-built TinyLlama MLC model directly — no upload needed.
Browse to pesnik.dev on a WebGPU browser to test.